In [1]:
%pip install pydicom
%pip install SimpleITK
%pip install git+https://github.com/Radiomics/pyradiomics.git
%pip install idc-index
%pip install pynrrd
%pip install dcmqi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 11.9 MB/s eta 0:00:00
  Cloning https://github.com/Radiomics/pyradiomics.git to /tmp/pip-req-build-up721jvt
  Running command git clone --filter=blob:none --quiet https://github.com/Radiomics/pyradiomics.git /tmp/pip-req-build-up721jvt
  Resolved https://github.com/Radiomics/pyradiomics.git to commit 8ed579383b44806651c463d5e691f3b2b57522ab
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 228.6 kB/s eta 0:00:00
  Created wheel for pyradiomics: filename=pyradiomics-3.1.1.dev111+g8ed579383-cp312-cp312-linux_x86_64.whl size=121732 sha256=858322e77a875da567c41ab806ca2edc2016065efcc8b11b98bc86ee9efaaac7
  Stored in directory: /tmp/pip-

In [2]:
import time
import pandas as pd
import pydicom
from idc_index import IDCClient
import os
import csv
import SimpleITK as sitk
import subprocess
import shutil
from google.colab import auth, files
from google.cloud import bigquery
import matplotlib.pyplot as plt
import nrrd
import numpy as np
from radiomics import featureextractor

In [3]:
BASE_DIR = "/content"
DOWNLOAD_DIR = os.path.join(BASE_DIR, "nlst")
COHORT_CSV = os.path.join(BASE_DIR, "patient_cohort.csv")
HEADERS_CSV = os.path.join(BASE_DIR, "ct_headers.csv")
FEATURES_CSV = os.path.join(BASE_DIR, "radiomic_features.csv")
TIMING_LOG = os.path.join(BASE_DIR, "timing_log.csv")

In [5]:
my_ProjectID = "nlst-radiomics"
os.environ["GCP_PROJECT_ID"] = my_ProjectID

auth.authenticate_user()
client = bigquery.Client(my_ProjectID)

lung_nodule_seg_series_descriptions = [
    'AIMI lung and nodule radiologist 5 corrected segmentation',
    'AIMI lung and nodule radiologist 4 corrected segmentation',
    'AIMI lung and nodule radiologist 8 corrected segmentation',
    'AIMI lung and nodule AI segmentation'
]

SR_description = "Sybil lesion bounding box"

query = f"""
SELECT
    DISTINCT seg.SeriesInstanceUID as SEG_SeriesInstanceUID,
    seg.SeriesDescription AS SEG_SeriesDescription,
    ct.SeriesInstanceUID as CT_SeriesInstanceUID,
    sr.SeriesInstanceUID as SR_SeriesInstanceUID,
    ct.StudyInstanceUID as StudyInstanceUID,
    ct.PatientID
FROM
    `bigquery-public-data.idc_v23.dicom_all` AS ct
JOIN
    `bigquery-public-data.idc_v23.dicom_all` AS seg
ON
    ct.PatientID = seg.PatientID
JOIN
    `bigquery-public-data.idc_v23.dicom_all` AS sr
ON
    ct.PatientID = sr.PatientID AND ct.StudyInstanceUID = sr.StudyInstanceUID
WHERE
    ct.collection_id = 'nlst'
    AND seg.collection_id = 'nlst'
    AND sr.collection_id = 'nlst'
    AND ct.Modality = 'CT'
    AND seg.Modality = 'SEG'
    AND sr.SeriesDescription = '{SR_description}'
    AND seg.SeriesDescription IN ({', '.join([f"'{s}'" for s in lung_nodule_seg_series_descriptions])})
    AND EXISTS (
        SELECT 1
        FROM UNNEST(seg.ReferencedSeriesSequence) as ref_seq
        WHERE ct.SeriesInstanceUID = ref_seq.SeriesInstanceUID
    )
"""

df = client.query(query).to_dataframe()

# Currently there are more SEG series than CT series; this is because some CT series have AI segmentation and radiologist corrected segmentation
# "Keep" the radiologist corrected segmentation by creating a priority column
ignore_desc = "AIMI lung and nodule AI segmentation"
df['Priority'] = (df['SEG_SeriesDescription'] == ignore_desc).astype(int)
df_sorted = df.sort_values(by=['CT_SeriesInstanceUID', 'Priority'])
df_clean = df_sorted.drop_duplicates(subset='CT_SeriesInstanceUID', keep='first')

final_df = df_clean[['PatientID', 'StudyInstanceUID', 'CT_SeriesInstanceUID', 'SEG_SeriesInstanceUID', 'SR_SeriesInstanceUID']]
final_df = final_df.sort_values(by=['PatientID', 'StudyInstanceUID'])
final_df.to_csv(COHORT_CSV, index=False)
final_df['NumNodules'] = 0
final_df['Status'] = "Unprocessed"
final_df.to_csv(COHORT_CSV, index=False)
print(f"PatientIDs and SeriesInstanceUIDs saved to {COHORT_CSV}")

PatientIDs and SeriesInstanceUIDs saved to /content/patient_cohort.csv


In [ ]:
def convert_CT_to_nrrd(pid, ct_uid, study_uid):
    try:
        reader = sitk.ImageSeriesReader()
        ct_series_dir = os.path.join(DOWNLOAD_DIR, pid, study_uid, "CT_" + ct_uid)
        dicom_names = reader.GetGDCMSeriesFileNames(ct_series_dir)
        reader.SetFileNames(dicom_names)
        ct_image = reader.Execute()
        output_path = os.path.join(DOWNLOAD_DIR, pid, study_uid, "CT.nrrd")
        sitk.WriteImage(ct_image, output_path)
    except Exception as e:
        print(f"Failed to convert CT to .nrrd for patient {pid}: {study_uid}: {e}")
        raise e

def convert_SEG_to_nrrd(pid, seg_uid, study_uid):
    seg_dir = os.path.join(DOWNLOAD_DIR, pid, study_uid, "SEG_" + seg_uid)
    seg_files = [f for f in os.listdir(seg_dir) if f.endswith('.dcm')]
    if not seg_files:
        raise FileNotFoundError(f"SEG DICOM not found in {seg_dir}")
    input_dicom = os.path.join(seg_dir, seg_files[0])
    output_dir = os.path.join(DOWNLOAD_DIR, pid, study_uid)
    command = [
        "segimage2itkimage",
        "--inputDICOM", input_dicom,
        "--outputDirectory", output_dir,
        "--outputType", "nrrd",
        "--prefix", "SEG"
    ]
    try:
        subprocess.run(command, check=True, capture_output=True, text=True)
        os.rename(os.path.join(output_dir, "SEG-1.nrrd"), os.path.join(output_dir, "lung.nrrd"))
        os.rename(os.path.join(output_dir, "SEG-2.nrrd"), os.path.join(output_dir, "nodules.nrrd"))
    except subprocess.CalledProcessError as e:
        print(f"SEG conversion failed for patient {pid}, study {study_uid}: {e.stderr}")
        raise e
    except FileNotFoundError as e:
        print(f"Renaming the .nrrd files failed for patient {pid}, study {study_uid}: {e}")
        raise e

In [6]:
pid = "100012"
study_uid = "1.2.840.113654.2.55.238034941445508011386463276954045956831"
ct_uid = "1.2.840.113654.2.55.335938848402215862539398120263494500079"
seg_uid = "1.2.276.0.7230010.3.1.3.17436516.3063603.1720665127.843589"
sr_uid = "1.2.826.0.1.3680043.8.498.37878940666999023000149218861683056435"

client = IDCClient()
client.download_from_selection(seriesInstanceUID=ct_uid, downloadDir=BASE_DIR)
client.download_from_selection(seriesInstanceUID=seg_uid, downloadDir=BASE_DIR)
client.download_from_selection(seriesInstanceUID=sr_uid, downloadDir=BASE_DIR)

In [9]:
sr_dir = os.path.join(DOWNLOAD_DIR, pid, study_uid, "SR_" + sr_uid)
sr_files = [f for f in os.listdir(sr_dir) if f.endswith('.dcm')]
sr_file = sr_files[0]
sr_path = os.path.join(sr_dir, sr_file)
sr = pydicom.dcmread(sr_path)
print(sr)

Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 214
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: Comprehensive 3D SR Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.2.826.0.1.3680043.8.498.75840729221877737722860494678577796679
(0002,0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002,0012) Implementation Class UID            UI: 1.2.826.0.1.3680043.9.7433.1.1
(0002,0013) Implementation Version Name         SH: 'highdicom0.27.0'
-------------------------------------------------
(0008,0012) Instance Creation Date              DA: '20251030'
(0008,0013) Instance Creation Time              TM: '233800.589699'
(0008,0016) SOP Class UID                       UI: Comprehensive 3D SR Storage
(0008,0018) SOP Instance UID                    UI: 1.2.826.0.1.3680043.8.498.75840729221877737722860494678577796679
(0008,0020) Study Date  

In [10]:
pid = "100012"
study_uid = "1.2.840.113654.2.55.3832109283939010833855886500020760484"
ct_uid = "1.2.840.113654.2.55.50761756412482430061802871163319122196"
seg_uid = "1.2.276.0.7230010.3.1.3.17436516.3063738.1720665129.785873"
sr_uid = "1.2.826.0.1.3680043.8.498.6547633571652463663967343716039950018"

client = IDCClient()
client.download_from_selection(seriesInstanceUID=ct_uid, downloadDir=BASE_DIR)
client.download_from_selection(seriesInstanceUID=seg_uid, downloadDir=BASE_DIR)
client.download_from_selection(seriesInstanceUID=sr_uid, downloadDir=BASE_DIR)

In [11]:
sr_dir = os.path.join(DOWNLOAD_DIR, pid, study_uid, "SR_" + sr_uid)
sr_files = [f for f in os.listdir(sr_dir) if f.endswith('.dcm')]
sr_file = sr_files[0]
sr_path = os.path.join(sr_dir, sr_file)
sr = pydicom.dcmread(sr_path)
print(sr)

Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 214
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: Comprehensive 3D SR Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.2.826.0.1.3680043.8.498.16544691815770055333019665811384559023
(0002,0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002,0012) Implementation Class UID            UI: 1.2.826.0.1.3680043.9.7433.1.1
(0002,0013) Implementation Version Name         SH: 'highdicom0.27.0'
-------------------------------------------------
(0008,0012) Instance Creation Date              DA: '20251031'
(0008,0013) Instance Creation Time              TM: '000057.702554'
(0008,0016) SOP Class UID                       UI: Comprehensive 3D SR Storage
(0008,0018) SOP Instance UID                    UI: 1.2.826.0.1.3680043.8.498.16544691815770055333019665811384559023
(0008,0020) Study Date  

In [12]:
pid = "100147"
study_uid = "1.2.840.113654.2.55.133032926508606330165457723341736723804"
ct_uid = "1.2.840.113654.2.55.247854884634057477137769379611681725965"
seg_uid = "1.2.276.0.7230010.3.1.3.17436516.3063942.1720665131.695761"
sr_uid = "1.2.826.0.1.3680043.8.498.43786955002590628477940364827943738848"

client = IDCClient()
client.download_from_selection(seriesInstanceUID=ct_uid, downloadDir=BASE_DIR)
client.download_from_selection(seriesInstanceUID=seg_uid, downloadDir=BASE_DIR)
client.download_from_selection(seriesInstanceUID=sr_uid, downloadDir=BASE_DIR)

In [13]:
sr_dir = os.path.join(DOWNLOAD_DIR, pid, study_uid, "SR_" + sr_uid)
sr_files = [f for f in os.listdir(sr_dir) if f.endswith('.dcm')]
sr_file = sr_files[0]
sr_path = os.path.join(sr_dir, sr_file)
sr = pydicom.dcmread(sr_path)
print(sr)

Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 214
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: Comprehensive 3D SR Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.2.826.0.1.3680043.8.498.17620509720668249378955585177617422673
(0002,0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002,0012) Implementation Class UID            UI: 1.2.826.0.1.3680043.9.7433.1.1
(0002,0013) Implementation Version Name         SH: 'highdicom0.27.0'
-------------------------------------------------
(0008,0012) Instance Creation Date              DA: '20251031'
(0008,0013) Instance Creation Time              TM: '001245.530077'
(0008,0016) SOP Class UID                       UI: Comprehensive 3D SR Storage
(0008,0018) SOP Instance UID                    UI: 1.2.826.0.1.3680043.8.498.17620509720668249378955585177617422673
(0008,0020) Study Date  